In [1]:
import json
import os
import time
# from prophet import Prophet
import pickle
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score,root_mean_squared_error
from statsmodels.tsa.statespace.sarimax import SARIMAX
import holidays
from xgboost import XGBRegressor
import numpy as np
import os
import logging
from pathlib import Path
import yaml
import mlflow
import pandas as pd
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
#directories
project_dir = Path().resolve()
data_dir = project_dir/"Data"
import holidays

c:\Users\Shaaf\Desktop\Data Science\Practice Projects\Transport Planning\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
print(f"Reading the resampled file")
df_resampled = pd.read_csv(data_dir/"hourly_demand.csv",parse_dates=True)
df_resampled['tpep_pickup_datetime'] = pd.to_datetime(df_resampled['tpep_pickup_datetime'])
df_resampled= df_resampled.sort_values(by='tpep_pickup_datetime')
print("Beginning data processing and Feature Enginnering")
holidays_usa=holidays.US()
df= df_resampled.copy()
df['Is_Holiday'] = df['tpep_pickup_datetime'].dt.date.apply(lambda d: d in holidays_usa).astype(int)
df['Day of the week']=df['tpep_pickup_datetime'].dt.day_of_week
df['Month']=df['tpep_pickup_datetime'].dt.month
df['Is weekend']=(df['Day of the week']>=5).astype(int)
df['Hour of the Day']=df['tpep_pickup_datetime'].dt.hour
lags = [1, 2, 3, 6, 12, 24, 48, 72, 168,336]
for i in lags:
    df[f"Lag{i}"]=df['Trips'].shift(i)
df = df.dropna(axis=0)
print(f"Shape of data before {df_resampled.shape}, shape after data processing {df.shape}")
print(f" The features are {df.columns}")
idx=int(len(df)*0.8)
train=df.iloc[:idx]
test=df.iloc[idx:] 
X_train=train.drop(columns=['Trips','tpep_pickup_datetime'],axis=1)
y_train=train['Trips']
X_test=test.drop(columns=['Trips','tpep_pickup_datetime'],axis=1)
y_test=test['Trips']

Reading the resampled file
Beginning data processing and Feature Enginnering
Shape of data before (12388, 2), shape after data processing (12052, 17)
 The features are Index(['tpep_pickup_datetime', 'Trips', 'Is_Holiday', 'Day of the week',
       'Month', 'Is weekend', 'Hour of the Day', 'Lag1', 'Lag2', 'Lag3',
       'Lag6', 'Lag12', 'Lag24', 'Lag48', 'Lag72', 'Lag168', 'Lag336'],
      dtype='object')


In [3]:
X_test

,Is_Holiday,Day of the week,Month,Is weekend,Hour of the Day,Lag1,Lag2,Lag3,Lag6,Lag12,Lag24,Lag48,Lag72,Lag168,Lag336
9977,0,4,2,0,13,760.0,608.0,529.0,410.0,161.0,632.0,598.0,581.0,689.0,600.0
9978,0,4,2,0,14,674.0,760.0,608.0,526.0,87.0,701.0,613.0,606.0,772.0,709.0
9979,0,4,2,0,15,741.0,674.0,760.0,537.0,57.0,662.0,711.0,599.0,835.0,769.0
9980,0,4,2,0,16,762.0,741.0,674.0,529.0,85.0,707.0,641.0,654.0,881.0,766.0
9981,0,4,2,0,17,811.0,762.0,741.0,608.0,130.0,849.0,896.0,741.0,948.0,947.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12383,0,6,5,1,19,660.0,737.0,715.0,660.0,168.0,832.0,979.0,823.0,526.0,671.0
12384,0,6,5,1,20,633.0,660.0,737.0,787.0,239.0,693.0,829.0,795.0,473.0,575.0
12385,0,6,5,1,21,533.0,633.0,660.0,717.0,403.0,770.0,749.0,1005.0,511.0,604.0
12386,0,6,5,1,22,509.0,533.0,633.0,715.0,484.0,911.0,921.0,951.0,442.0,542.0


In [16]:
yaml_path = project_dir/'config.yaml'

with open(yaml_path, "r") as file:
    config = yaml.safe_load(file)

In [17]:
config

{'resample_type': 'H',
 'sarima': {'order': [1, 1, 0],
  'seasonal_order': [1, 0, 1, 24],
  'enforce_stationarity': False,
  'enforce_invertibility': False},
 'best_params': {'n_estimators': 603,
  'max_depth': 3,
  'learning_rate': 0.19,
  'reg_alpha': 0.9,
  'reg_lambda': 0.64}}

In [18]:
model = XGBRegressor(**config['best_params'])

In [19]:
model.fit(X_train,y_train)

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'reg:squarederror'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes

In [20]:
predicted = model.predict(X_test)
mae=mean_absolute_error(y_test,predicted)
mse=mean_squared_error(y_test,predicted)
rmse=root_mean_squared_error(y_test,predicted)
r2 = r2_score(y_test,predicted)
metrics= {
    'MAE':mae,
    'MSE':mse,
    'RMSE':rmse,
    'r2_score':r2
    }

In [21]:
metrics

{'MAE': 35.56037902832031,
 'MSE': 2589.49658203125,
 'RMSE': 50.8870964050293,
 'r2_score': 0.9695823788642883}

In [ ]:
# {'MAE': 36.8072395324707,
#  'MSE': 2750.69482421875,
#  'RMSE': 52.44706726074219,
#  'r2_score': 0.967688798904419}